In [ ]:
!pip install librosa pandas tqdm pyarrow

In [ ]:
# ==============================================================================
# BLOCK 1: LIBRARIES AND DEPENDENCIES
# ==============================================================================
import os
import hashlib
import zipfile
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from google.colab import drive

# ==============================================================================
# BLOCK 2: MANUAL TEXT-BASED DICTIONARY
# ==============================================================================

MANUAL_DICTIONARY = {
    "A Million Dreams": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "en"},
    "Bad Day": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "en"},
    "Bao Tien Mot Mo Binh Yen": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "vi"},
    "Black Water Lilies": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "en"},
    "Buc Tranh va Canh Chim": {"genre": "ballad", "lyrics_theme": "motivation", "language_code": "vi"},
    "Daylight": {"genre": "cyberpunk", "lyrics_theme": "loneliness", "language_code": "en"},
    "Drag Path": {"genre": "cyberpunk", "lyrics_theme": "escapism", "language_code": "en"},
    "End of Beginning": {"genre": "ballad", "lyrics_theme": "escapism", "language_code": "en"},
    "First And Last": {"genre": "ballad", "lyrics_theme": "heartbreak", "language_code": "ko"},
    "From Now On": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "en"},
    "Giu Anh Cho Ngay Hom Qua": {"genre": "ballad", "lyrics_theme": "motivation", "language_code": "vi"},
    "Golden": {"genre": "cyberpunk", "lyrics_theme": "general", "language_code": "en"},
    "Grandma's Home": {"genre": "alternative", "lyrics_theme": "comfort", "language_code": "zxx"},
    "Hen Lan Sau": {"genre": "ballad", "lyrics_theme": "heartbreak", "language_code": "vi"},
    "Kiem Dau Bay Gio": {"genre": "ballad", "lyrics_theme": "heartbreak", "language_code": "vi"},
    "Lemon": {"genre": "cyberpunk", "lyrics_theme": "general", "language_code": "ja"},
    "Let Down": {"genre": "cyberpunk", "lyrics_theme": "escapism", "language_code": "en"},
    "My old story": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "ko"},
    "Nguoi O Dung Ve": {"genre": "cyberpunk", "lyrics_theme": "general", "language_code": "vi"},
    "nop": {"genre": "alternative", "lyrics_theme": "comfort", "language_code": "zxx"},
    "Rewrite The Stars": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "en"},
    "Sailor Song": {"genre": "ballad", "lyrics_theme": "escapism", "language_code": "en"},
    "Start Over": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "ko"},
    "Survive": {"genre": "cyberpunk", "lyrics_theme": "loneliness", "language_code": "en"},
    "The Nights": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "en"},
    "This Is Me": {"genre": "cyberpunk", "lyrics_theme": "motivation", "language_code": "en"},
    "Ve Nha": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "vi"},
    "Waiting For Love": {"genre": "cyberpunk", "lyrics_theme": "comfort", "language_code": "en"},
    "Young and Beautiful": {"genre": "ballad", "lyrics_theme": "heartbreak", "language_code": "en"},
    "yung kai": {"genre": "ballad", "lyrics_theme": "comfort", "language_code": "ko"}
}

"""
Keyword for dictionary
genre: ballad, cyberpunk
lyrics_theme: General, Comfort, Motivation, Loneliness, Heartbreak, Escapism
language_code: iso codes
"""

# ==============================================================================
# BLOCK 3: AUTOMATED AUDIO & NUMBER TRANSLATION ENGINES (BALANCED MATRIX)
# ==============================================================================
def calculate_advanced_audio_features(y, sr):
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo_bpm = float(tempo)

    energy_rms = float(np.mean(librosa.feature.rms(y=y)))
    spectral_flatness = float(np.mean(librosa.feature.spectral_flatness(y=y)))

    # FIX 1: Use np.tanh to strictly scale variance into a clean 0.0 - 1.0 range
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    raw_variance = float(np.var(onset_env))
    danceability_proxy = float(np.tanh(raw_variance))

    y_harmonic, y_percussive = librosa.effects.hpss(y)
    harmonic_energy = np.sum(y_harmonic ** 2)
    percussive_energy = np.sum(y_percussive ** 2) + 1e-6
    instrumentalness = float(harmonic_energy / (harmonic_energy + percussive_energy))

    return tempo_bpm, energy_rms, spectral_flatness, danceability_proxy, instrumentalness


def map_sentiment_from_theme(lyrics_theme):
    theme_lower = lyrics_theme.lower()
    if theme_lower == "comfort":
        return 0.65
    elif theme_lower == "motivation":
        return 0.85
    elif theme_lower == "loneliness":
        return -0.70
    elif theme_lower == "heartbreak":
        return -0.55
    elif theme_lower == "escapism":
        return -0.15
    else:
        return 0.0


def calculate_music_valence(tempo, energy, danceability, genre):
    # Pure acoustic-based brightness computation (independent of lyrics)
    base_valence = (danceability * 0.5) + (min(tempo, 140.0) / 140.0 * 0.5)

    # Reasonable genre damping modifiers
    genre_lower = genre.lower()
    if "lofi" in genre_lower or "ballad" in genre_lower:
        base_valence *= 0.7  # Minor, soft scale
    elif "cyberpunk" in genre_lower or "rock" in genre_lower:
        base_valence *= 0.65 # Aggressive, heavy scale

    return float(np.clip(base_valence, 0.0, 1.0))


def determine_granular_mood(energy, music_valence, lyrics_sentiment):
    # FIX 2: Re-engineered Multimodal Matrix to empower lyrics_sentiment
    # Threshold for High Arousal (Active/Vibrant music state)
    if energy >= 0.07:
        if lyrics_sentiment >= 0.2:
            # High energy + Happy lyrics -> Euphoric if music is bright, else Tense/Hype
            return "euphoric" if music_valence >= 0.40 else "tense/hype"
        elif lyrics_sentiment <= -0.2:
            # High energy + Sad lyrics -> Perfect Paradox (Bitter-Sweet)
            return "bitter-sweet / happy-sad"
        else:
            return "tense/hype"

    # Threshold for Low Arousal (Calm/Quiet music state)
    else:
        if lyrics_sentiment >= 0.2:
            return "peaceful"
        elif lyrics_sentiment <= -0.2:
            return "gloomy"
        else:
            return "peaceful" if music_valence >= 0.45 else "gloomy"

# ==============================================================================
# BLOCK 4: RECURSIVE PIPELINE EXECUTION (UPDATED PARAMETERS)
# ==============================================================================
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/music.zip"
EXTRACT_DIR = "/content/raw_music"

print("Extracting seed dataset")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

audio_file_paths = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith('.mp3'):
            audio_file_paths.append(os.path.join(root, file))

print(f"Found {len(audio_file_paths)} audio tracks.")

metadata_records = []

for file_path in tqdm(audio_file_paths):
    filename = os.path.basename(file_path)

    try:
        clean_name = filename.replace(".mp3", "").replace(".MP3", "")
        if " - " in clean_name:
            track_title, artist = clean_name.split(" - ", 1)
            track_title = track_title.strip()
            artist = artist.strip()
        else:
            track_title = clean_name.strip()
            artist = "Unknown"

        track_id = str(hashlib.md5(filename.encode('utf-8')).hexdigest())

        # FETCH FROM QUAN'S DICTIONARY
        user_mapped_data = MANUAL_DICTIONARY.get(
            track_title,
            {"genre": "alternative", "lyrics_theme": "general", "language_code": "en"}
        )
        genre = str(user_mapped_data["genre"])
        lyrics_theme = str(user_mapped_data["lyrics_theme"])
        language_code = str(user_mapped_data["language_code"])

        # AUTOMATED NUMERICAL CONVERSIONS
        lyrics_sentiment = map_sentiment_from_theme(lyrics_theme)

        # LIBROSA AUDIO FEATURES
        y, sr = librosa.load(file_path, sr=None)
        tempo, energy, flatness, dance, instrumental = calculate_advanced_audio_features(y, sr)

        # SMART MOOD LOGIC GENERATION
        # FIXED LINE: Only pass 4 acoustic arguments now!
        music_valence = calculate_music_valence(tempo, energy, dance, genre)

        granular_mood = str(determine_granular_mood(energy, music_valence, lyrics_sentiment))

        record = {
            "track_id": track_id,
            "track_title": track_title,
            "artist": artist,
            "genre": genre,
            "language_code": language_code,
            "tempo_bpm": round(float(tempo), 1),
            "energy_rms": round(float(energy), 4),
            "spectral_flatness": round(float(flatness), 4),
            "danceability_proxy": round(float(dance), 4),
            "instrumentalness": round(float(instrumental), 4),
            "music_valence": round(float(music_valence), 4),
            "lyrics_theme": lyrics_theme,
            "lyrics_sentiment": round(float(lyrics_sentiment), 4),
            "granular_mood": granular_mood
        }
        metadata_records.append(record)

    except Exception as e:
        print(f"Error processing file {filename}: {str(e)}")
        continue

# ==============================================================================
# BLOCK 5: DATA EXPORT AND VALIDATION
# ==============================================================================
if len(metadata_records) == 0:
    print("PIPELINE FAILED: No audio records created.")
else:
    df_final = pd.DataFrame(metadata_records)

    schema_mapping = {
        'track_id': 'string', 'track_title': 'string', 'artist': 'string',
        'genre': 'string', 'language_code': 'string', 'lyrics_theme': 'string', 'granular_mood': 'string',
        'tempo_bpm': 'float64', 'energy_rms': 'float64', 'spectral_flatness': 'float64',
        'danceability_proxy': 'float64', 'instrumentalness': 'float64', 'music_valence': 'float64', 'lyrics_sentiment': 'float64'
    }
    df_final = df_final.astype(schema_mapping)

    OUTPUT_PARQUET = "/content/drive/MyDrive/music_metadata.parquet"
    df_final.to_parquet(OUTPUT_PARQUET, index=False, engine='pyarrow')